In [17]:
import torch
from torch import nn, optim
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

In [18]:
class Actor(nn.Module):
	def __init__(self, memory_dim, state_dim, action_dim, hidden_dim):
		super().__init__()

		self.action_dim = action_dim

		# Projects masked state into memory space
		self.encoder = nn.Sequential(
			nn.Linear(state_dim, memory_dim),
			nn.Tanh(),
		)

		# Gates memory with encoded state, outputs action + memory delta
		self.decoder = nn.Sequential(
			nn.Linear(memory_dim, hidden_dim),
			nn.LeakyReLU(),
			nn.Linear(hidden_dim, hidden_dim),
			nn.LeakyReLU(),
			nn.Linear(hidden_dim, memory_dim + action_dim),
		)

	def forward(self, memory, state, alpha, sigma):
		x = self.encoder(state)
		x = self.decoder(memory * x)

		# tanh bounds action to (-1, 1) to match MountainCarContinuous action space
		# sigma is the action exploration noise — set to 0 during critic/target updates
		action = torch.tanh(x[..., :self.action_dim])
		action_noise = sigma * torch.randn_like(action).clip(min=-2.5, max=2.5)
		action = (action + action_noise).clip(-1.0, 1.0)

		# Memory update: deterministic — shaped only by diversity regularization, no noise
		delta_memory = x[..., self.action_dim:]
		delta_memory_norm = torch.linalg.vector_norm(delta_memory, dim=-1, keepdim=True)
		delta_memory_scale = delta_memory_norm.clip(max=alpha)
		delta_memory = delta_memory_scale * delta_memory / (delta_memory_norm + 1e-7)
		next_memory = (memory + delta_memory).clip(min=0, max=1)

		return action, next_memory, delta_memory

	@staticmethod
	def get_diversity(delta_memory):
		"""Mean per-dimension variance of delta_memory across the batch.
		Higher = more diverse memory updates across samples (exploration signal)."""
		return delta_memory.var(dim=0).mean()


class Critic(nn.Module):
	def __init__(self, memory_dim, state_dim, action_dim, hidden_dim):
		super().__init__()

		self.ffn = nn.Sequential(
			nn.Linear(2 * memory_dim + state_dim + action_dim, hidden_dim),
			nn.LeakyReLU(),
			nn.Linear(hidden_dim, hidden_dim),
			nn.LeakyReLU(),
			nn.Linear(hidden_dim, hidden_dim),
			nn.LeakyReLU(),
			nn.Linear(hidden_dim, 1),
		)

	def forward(self, memory, state, action, next_memory):
		# Critic sees both memory and next_memory, giving it the full memory transition
		x = torch.cat([memory, state, action, next_memory], dim=-1)
		return self.ffn(x)

In [19]:
class ReplayBuffer:
	"""Pre-allocated numpy ring buffer — avoids per-step tuple allocation and slow random.sample.
	torch.from_numpy gives zero-copy CPU tensors; .to(device) is then a single contiguous transfer."""

	def __init__(self, capacity, memory_dim, state_dim, action_dim):
		self.capacity = capacity
		self.ptr  = 0
		self.size = 0
		self.memories      = np.zeros((capacity, memory_dim), dtype=np.float32)
		self.states        = np.zeros((capacity, state_dim),  dtype=np.float32)
		self.actions       = np.zeros((capacity, action_dim), dtype=np.float32)
		self.next_memories = np.zeros((capacity, memory_dim), dtype=np.float32)
		self.next_states   = np.zeros((capacity, state_dim),  dtype=np.float32)
		self.rewards       = np.zeros((capacity, 1),          dtype=np.float32)
		self.dones         = np.zeros((capacity, 1),          dtype=np.float32)

	def push(self, memory, state, action, next_memory, next_state, reward, done):
		i = self.ptr
		self.memories[i]      = memory
		self.states[i]        = state
		self.actions[i]       = action
		self.next_memories[i] = next_memory
		self.next_states[i]   = next_state
		self.rewards[i]       = reward
		self.dones[i]         = done
		self.ptr  = (self.ptr + 1) % self.capacity
		self.size = min(self.size + 1, self.capacity)

	def sample(self, batch_size, device):
		# np.random.randint is much faster than random.sample on a deque
		idx = np.random.randint(0, self.size, size=batch_size)
		# torch.from_numpy: zero-copy until .to(device) — avoids redundant allocation
		return (
			torch.from_numpy(self.memories[idx]).to(device),
			torch.from_numpy(self.states[idx]).to(device),
			torch.from_numpy(self.actions[idx]).to(device),
			torch.from_numpy(self.next_memories[idx]).to(device),
			torch.from_numpy(self.next_states[idx]).to(device),
			torch.from_numpy(self.rewards[idx]).to(device),
			torch.from_numpy(self.dones[idx]).to(device),
		)

	def __len__(self):
		return self.size

In [23]:
# ── Environment ────────────────────────────────────────────────────────────────
env = gym.make("MountainCarContinuous-v0")

# obs = [position, velocity]  →  velocity masked, so the memory must learn to track it
STATE_DIM   = 1
ACTION_DIM  = env.action_space.shape[0]   # 1
MEMORY_DIM  = 8
HIDDEN_DIM  = 256

# ── TD3 hyperparameters ────────────────────────────────────────────────────────
LR           = 3e-4
GAMMA        = 0.99
TAU          = 0.005
BATCH_SIZE   = 256
BUFFER_SIZE  = 10_000
POLICY_DELAY = 2

# ── Actor / noise hyperparameters ─────────────────────────────────────────────
ALPHA        = 0.15   # max norm of delta_memory per step
SIGMA        = 0.3    # action exploration noise (rollout)
TARGET_SIGMA = 0.2    # action smoothing noise for target policy (TD3)

# ── Diversity (exploration) loss ───────────────────────────────────────────────
LAMBDA_DIV = 0.005

# ── Training schedule ──────────────────────────────────────────────────────────
MAX_EPISODES = 400
MAX_STEPS    = 999
WARMUP_STEPS = 5_000

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}  |  state_dim={STATE_DIM} (velocity masked)  |  memory_dim={MEMORY_DIM}")

Device: cuda  |  state_dim=1 (velocity masked)  |  memory_dim=8


In [24]:
def soft_update(net, target_net, tau):
	with torch.no_grad():
		for p, tp in zip(net.parameters(), target_net.parameters()):
			tp.data.copy_(tau * p.data + (1.0 - tau) * tp.data)


def make_network(cls, *args):
	net    = cls(*args).to(DEVICE)
	target = cls(*args).to(DEVICE)
	target.load_state_dict(net.state_dict())
	return net, target


# Actor + target on GPU for training
actor,   actor_target   = make_network(Actor,  MEMORY_DIM, STATE_DIM, ACTION_DIM, HIDDEN_DIM)
critic1, critic1_target = make_network(Critic, MEMORY_DIM, STATE_DIM, ACTION_DIM, HIDDEN_DIM)
critic2, critic2_target = make_network(Critic, MEMORY_DIM, STATE_DIM, ACTION_DIM, HIDDEN_DIM)

# ── CPU actor for rollout ──────────────────────────────────────────────────────
# Single-sample GPU inference has high kernel-launch overhead relative to compute.
# A CPU copy of the actor avoids that for the per-step rollout (batch size = 1).
actor_cpu = Actor(MEMORY_DIM, STATE_DIM, ACTION_DIM, HIDDEN_DIM).cpu()
actor_cpu.load_state_dict({k: v.cpu() for k, v in actor.state_dict().items()})

def sync_actor_cpu():
	"""Pull latest weights from GPU actor into the CPU rollout copy."""
	with torch.no_grad():
		for p_cpu, p_gpu in zip(actor_cpu.parameters(), actor.parameters()):
			p_cpu.data.copy_(p_gpu.data.cpu())

actor_opt   = optim.Adam(actor.parameters(),   lr=LR)
critic1_opt = optim.Adam(critic1.parameters(), lr=LR)
critic2_opt = optim.Adam(critic2.parameters(), lr=LR)

replay_buffer = ReplayBuffer(BUFFER_SIZE, MEMORY_DIM, STATE_DIM, ACTION_DIM)

print(f"Actor params:   {sum(p.numel() for p in actor.parameters()):,}")
print(f"Critic params:  {sum(p.numel() for p in critic1.parameters()):,} (x2)")

Actor params:   70,425
Critic params:  136,705 (x2)


In [26]:
episode_rewards = []
actor_losses    = []
diversity_vals  = []

total_steps    = 0
critic_updates = 0

for episode in range(MAX_EPISODES):
	obs, _ = env.reset()

	# ── Mask velocity: keep only position (obs[0]) ──────────────────────────────
	state  = obs[:1].astype(np.float32)                      # shape (1,)
	memory = np.full(MEMORY_DIM, 0.5, dtype=np.float32)      # shape (memory_dim,)

	episode_reward = 0.0

	for step in range(MAX_STEPS):
		total_steps += 1

		# ── Rollout on CPU actor (avoids per-step GPU kernel launch overhead) ────
		with torch.no_grad():
			state_t  = torch.from_numpy(state).unsqueeze(0)   # CPU tensor, shape (1, 1)
			memory_t = torch.from_numpy(memory).unsqueeze(0)  # CPU tensor, shape (1, mem_dim)

			if total_steps < WARMUP_STEPS:
				# Random action; run CPU actor with sigma=0 to coherently evolve memory
				action = env.action_space.sample()                           # shape (1,)
				_, next_memory_t, _ = actor_cpu(memory_t, state_t, ALPHA, 0.0)
			else:
				# sigma > 0: Actor adds exploration noise to action only (not memory)
				action_t, next_memory_t, _ = actor_cpu(memory_t, state_t, ALPHA, SIGMA)
				action = action_t.squeeze(0).numpy()                         # shape (1,)

			next_memory = next_memory_t.squeeze(0).numpy()                   # shape (memory_dim,)

		# ── Environment step ─────────────────────────────────────────────────────
		next_obs, reward, terminated, truncated, _ = env.step(action)
		done = terminated or truncated
		next_state = next_obs[:1].astype(np.float32)                         # mask velocity

		# ── Store transition ─────────────────────────────────────────────────────
		replay_buffer.push(memory, state, action, next_memory, next_state, reward, float(done))

		state  = next_state
		memory = next_memory
		episode_reward += reward

		# ── Learning update ──────────────────────────────────────────────────────
		if len(replay_buffer) >= BATCH_SIZE and total_steps >= WARMUP_STEPS:
			mem_b, state_b, action_b, next_mem_b, next_state_b, reward_b, done_b = \
				replay_buffer.sample(BATCH_SIZE, DEVICE)

			# ── Critic update ────────────────────────────────────────────────────
			with torch.no_grad():
				# Target policy smoothing via TARGET_SIGMA passed through Actor.sigma
				next_action_t, next_next_mem_t, _ = actor_target(
					next_mem_b, next_state_b, ALPHA, TARGET_SIGMA
				)
				next_action_t = next_action_t.clip(-1.0, 1.0)
				q1_tgt = critic1_target(next_mem_b, next_state_b, next_action_t, next_next_mem_t)
				q2_tgt = critic2_target(next_mem_b, next_state_b, next_action_t, next_next_mem_t)
				q_tgt  = reward_b + GAMMA * (1.0 - done_b) * torch.min(q1_tgt, q2_tgt)

			q1_pred = critic1(mem_b, state_b, action_b, next_mem_b)
			q2_pred = critic2(mem_b, state_b, action_b, next_mem_b)

			# Single backward for both critics — avoids a second full graph traversal
			c_loss = nn.functional.mse_loss(q1_pred, q_tgt) + \
					 nn.functional.mse_loss(q2_pred, q_tgt)
			critic1_opt.zero_grad()
			critic2_opt.zero_grad()
			c_loss.backward()
			critic1_opt.step()
			critic2_opt.step()

			critic_updates += 1

			# ── Delayed actor update ─────────────────────────────────────────────
			if critic_updates % POLICY_DELAY == 0:
				# sigma=0: clean deterministic action; delta_memory shaped by diversity loss only
				action_pred, next_mem_pred, delta_mem_pred = actor(mem_b, state_b, ALPHA, 0.0)

				q_actor   = critic1(mem_b, state_b, action_pred, next_mem_pred)
				diversity = Actor.get_diversity(delta_mem_pred)

				# Minimise loss = maximise Q + maximise memory diversity
				actor_loss = -q_actor.mean() - LAMBDA_DIV * diversity

				actor_opt.zero_grad()
				actor_loss.backward()
				actor_opt.step()

				# Soft-update GPU target networks
				soft_update(actor,   actor_target,   TAU)
				soft_update(critic1, critic1_target, TAU)
				soft_update(critic2, critic2_target, TAU)

				# Sync CPU rollout actor with updated GPU weights
				sync_actor_cpu()

				actor_losses.append(actor_loss.item())
				diversity_vals.append(diversity.item())

		if done:
			break

	episode_rewards.append(episode_reward)

	if (episode + 1) % 10 == 0:
		avg10 = np.mean(episode_rewards[-10:])
		avg_div = np.mean(diversity_vals[-100:]) if diversity_vals else 0.0
		print(f"Ep {episode+1:4d} | reward {episode_reward:8.1f} | avg10 {avg10:8.1f} "
			  f"| steps {total_steps:7d} | diversity {avg_div:.4f}")

print("\nTraining complete!")

Ep   10 | reward    -48.1 | avg10    -29.9 | steps    9205 | diversity 0.0023
Ep   20 | reward    -77.5 | avg10    -11.6 | steps   16771 | diversity 0.0022
Ep   30 | reward    -68.7 | avg10    -58.6 | steps   26256 | diversity 0.0022
Ep   40 | reward     83.0 | avg10     42.6 | steps   30981 | diversity 0.0025
Ep   50 | reward    -63.9 | avg10    -19.5 | steps   38879 | diversity 0.0026
Ep   60 | reward    -67.0 | avg10    -25.1 | steps   48090 | diversity 0.0024
Ep   70 | reward    -79.0 | avg10    -69.2 | steps   58080 | diversity 0.0018
Ep   80 | reward    -72.8 | avg10    -77.3 | steps   68070 | diversity 0.0024
Ep   90 | reward     76.6 | avg10    -45.8 | steps   77126 | diversity 0.0023
Ep  100 | reward    -79.5 | avg10    -75.1 | steps   87116 | diversity 0.0024
Ep  110 | reward    -76.1 | avg10    -40.0 | steps   96370 | diversity 0.0024
Ep  120 | reward    -65.1 | avg10    -73.1 | steps  106360 | diversity 0.0025
Ep  130 | reward    -70.5 | avg10    -33.4 | steps  115371 | div

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.plot(episode_rewards, alpha=0.4, color='steelblue', label='Episode reward')
window = 20
if len(episode_rewards) >= window:
	ma = np.convolve(episode_rewards, np.ones(window) / window, mode='valid')
	ax.plot(range(window - 1, len(episode_rewards)), ma, color='steelblue',
			linewidth=2, label=f'{window}-ep avg')
ax.axhline(90, color='green', linestyle='--', alpha=0.6, label='Solved threshold')
ax.set_xlabel('Episode'); ax.set_ylabel('Total reward')
ax.set_title('MountainCarContinuous (velocity masked)'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(actor_losses, alpha=0.5, color='crimson')
ax.set_xlabel('Actor update'); ax.set_ylabel('Actor loss')
ax.set_title('Actor Loss  (-Q - λ·diversity)'); ax.grid(alpha=0.3)

ax = axes[2]
ax.plot(diversity_vals, alpha=0.5, color='darkorange')
ax.set_xlabel('Actor update'); ax.set_ylabel('Var(delta_memory).mean()')
ax.set_title('Memory Update Diversity'); ax.grid(alpha=0.3)

plt.suptitle('Memory Actor + TD3 — Velocity-Masked MountainCar', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()